<a href="https://colab.research.google.com/github/debashisdotchatterjee/CAPE_MD_1/blob/main/CAPE_MD_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q torch-geometric pandas scipy scikit-learn tqdm tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00


In [3]:
!pip install -q torch-geometric

import torch
import torch_geometric

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch Geometric:", torch_geometric.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.0 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
PyTorch Geometric: 2.8.0


In [5]:
!python CAPE_MD_complete_verification.py

Device: cuda PyTorch: 2.11.0+cu128 Fast mode: True
Synthetic shapes: {'pos': (1399, 6, 3), 'energy': (1399,), 'force': (1399, 6, 3), 'z': (6,)}
Synthetic curvature pairs: 1200
Training DirectForceNet: 100% 30/30 [00:05<00:00,  5.94it/s]
Training ConservativeNet: 100% 30/30 [00:06<00:00,  4.63it/s]
Training CAPE-MD: 100% 30/30 [00:10<00:00,  2.76it/s]
Figure(850x520)
Saved: CAPE_MD_RESULTS/figures/synthetic_training.png
<IPython.core.display.Markdown object>
Saved: CAPE_MD_RESULTS/tables/synthetic_pointwise.csv
<IPython.core.display.Markdown object>
Saved: CAPE_MD_RESULTS/tables/synthetic_curvature.csv
Figure(1500x450)
Saved: CAPE_MD_RESULTS/figures/synthetic_force_scatter.png
<IPython.core.display.Markdown object>
Saved: CAPE_MD_RESULTS/tables/synthetic_rollout.csv
Figure(850x520)
Saved: CAPE_MD_RESULTS/figures/synthetic_rollout_drift.png
<IPython.core.display.Markdown object>
• Energy_MAE: lowest observed value = CAPE-MD (0.48781).
• Energy_RMSE: lowest observed value = CAPE-MD (0.684

In [6]:
from pathlib import Path

script_path = Path("/content/CAPE_MD_complete_verification.py")
text = script_path.read_text(encoding="utf-8")

old = """from torch_geometric.datasets import MD17
    ds=MD17(root='data/MD17',name=CFG.real_dataset_name); print(ds,ds[0])"""

new = """from torch_geometric.datasets import MD17

    # PyTorch Geometric 2.8.0 still uses an obsolete Materials Cloud URL.
    # Patch it to the current official rMD17 archive.
    MD17.revised_url = (
        "https://archive.materialscloud.org/records/"
        "pfffs-fff86/files/rmd17.tar.bz2?download=1"
    )

    ds=MD17(root='data/MD17',name=CFG.real_dataset_name); print(ds,ds[0])"""

if old not in text:
    print("Exact original block was not found; applying a line-based patch.")

    old_line = "from torch_geometric.datasets import MD17"
    replacement = """from torch_geometric.datasets import MD17

    # Current official Materials Cloud rMD17 archive:
    MD17.revised_url = (
        "https://archive.materialscloud.org/records/"
        "pfffs-fff86/files/rmd17.tar.bz2?download=1"
    )"""

    if "pfffs-fff86/files/rmd17.tar.bz2" not in text:
        text = text.replace(old_line, replacement, 1)
else:
    text = text.replace(old, new, 1)

script_path.write_text(text, encoding="utf-8")

print("Script patched successfully.")

Exact original block was not found; applying a line-based patch.
Script patched successfully.


In [7]:
!grep -n -A8 -B2 "pfffs-fff86" /content/CAPE_MD_complete_verification.py

371-    MD17.revised_url = (
372-        "https://archive.materialscloud.org/records/"
373:        "pfffs-fff86/files/rmd17.tar.bz2?download=1"
374-    )
375-    import torch_geometric
376-    ds=MD17(root='data/MD17',name=CFG.real_dataset_name); print(ds,ds[0])
377-    pos=[]; ene=[]; frc=[]; z=ds[0].z.long()
378-    for item in tqdm(ds,desc='Reading rMD17'):
379-        pos.append(item.pos.float()); ene.append((item.energy if hasattr(item,'energy') and item.energy is not None else item.y).reshape(-1)[0].float()); frc.append((item.force if hasattr(item,'force') and item.force is not None else item.dy).float())
380-    real={'pos':torch.stack(pos),'energy':torch.stack(ene),'force':torch.stack(frc),'z':z}; print({k:tuple(v.shape) for k,v in real.items()})
381-    def blocked(total,counts,spacing,gap):


In [8]:
!rm -rf /content/data/MD17

In [9]:
!python /content/CAPE_MD_complete_verification.py

Device: cuda PyTorch: 2.11.0+cu128 Fast mode: True
Synthetic shapes: {'pos': (1399, 6, 3), 'energy': (1399,), 'force': (1399, 6, 3), 'z': (6,)}
Synthetic curvature pairs: 1200
Training DirectForceNet: 100% 30/30 [00:05<00:00,  5.91it/s]
Training ConservativeNet: 100% 30/30 [00:06<00:00,  4.69it/s]
Training CAPE-MD: 100% 30/30 [00:10<00:00,  2.79it/s]
Figure(850x520)
Saved: CAPE_MD_RESULTS/figures/synthetic_training.png
<IPython.core.display.Markdown object>
Saved: CAPE_MD_RESULTS/tables/synthetic_pointwise.csv
<IPython.core.display.Markdown object>
Saved: CAPE_MD_RESULTS/tables/synthetic_curvature.csv
Figure(1500x450)
Saved: CAPE_MD_RESULTS/figures/synthetic_force_scatter.png
<IPython.core.display.Markdown object>
Saved: CAPE_MD_RESULTS/tables/synthetic_rollout.csv
Figure(850x520)
Saved: CAPE_MD_RESULTS/figures/synthetic_rollout_drift.png
<IPython.core.display.Markdown object>
• Energy_MAE: lowest observed value = CAPE-MD (0.48781).
• Energy_RMSE: lowest observed value = CAPE-MD (0.684